# Risk-averse mutation analysis — contract-period FE distributions

This notebook reads `Hourly_FE_Index.csv` and the corresponding contract-period FE CSVs from each risk setting. It is intended for match-level distribution diagnostics after the risk-averse mutation simulation has saved FE exports.

The simulation notebook retained the `Hourly_FE` folder name for compatibility, but the rows are contract-period financial exposure by replication, not hourly observations.

In [ ]:
# ============================================================
# SECTION 1. USER CONTROLS
# ============================================================
from pathlib import Path

# Path controls.
code_root_override = None
risk_averse_mutation_dir_override = None
risk_output_base_dir_override = None
risk_averse_mutation_folder_name = "simulation_mutation_averse"
risk_output_base_folder_name = "Output files (Risk Averse, Mutation, Verified)"
risk_output_index_filename = "Risk_Averse_Dynamic_Output_Index.csv"
analysis_output_folder_name = "Risk_Averse_Mutation_Analysis"

# Required match/scenario controls.
selected_match_id = 1
selected_mutation_family = "cannibalization"   # "shape", "basis", "load_price", "cannibalization", or None for all
selected_target_physical_shifts = [-0.10, -0.30, -0.50]  # mutation scenarios; baseline is added if include_baseline=True
include_baseline = True

# Risk filters.
selected_risk_labels = None                     # e.g. ["joint_low", "joint_medium"]
selected_risk_modes = None    # e.g. ["Seller risk-averse"], None
selected_lambda_s_values = None
selected_lambda_b_values = None

# FE metric controls.
fe_party_to_plot = "both"                       # "seller", "buyer", or "both"
fe_columns = {"seller": "seller_fe", "buyer": "buyer_fe"}
revenue_column = "seller_revenue"

# Output controls.
save_fe_combined_table = True
save_fe_summary_table = True

In [ ]:
# ============================================================
# SECTION 2. FIGURE CONTROLS
# ============================================================
make_figures = True
save_png = True
save_pdf = False
save_svg = False
display_figures_in_notebook = True
figure_dpi = 600

# Figure switches.
make_boxplot_figure = True
make_ecdf_figure = True
make_mean_std_figure = True

# Layout and style.
default_fig_width = 12.0
default_fig_height = 5.2
axis_label_fontsize = 11
tick_label_fontsize = 10
panel_title_fontsize = 12
legend_fontsize = 9
x_tick_rotation = 35
boxplot_show_fliers = False
ecdf_linewidth = 1.8

family_display_labels = {
    "shape": "Profile-shape deterioration",
    "basis": "Basis deterioration",
    "load_price": "Buyer load-price intensification",
    "cannibalization": "Seller-side cannibalization",
    "baseline": "Baseline",
}
risk_mode_order = ["Seller risk-averse", "Buyer risk-averse", "Joint risk-averse", "Risk neutral", "Other"]

In [ ]:
# ============================================================
# SECTION 3. IMPORTS, PATHS, AND RISK OUTPUT DISCOVERY
# ============================================================
from __future__ import annotations

import re
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

NOTEBOOK_CWD = Path.cwd().resolve()


def _as_path_or_none(value) -> Optional[Path]:
    if value is None:
        return None
    text = str(value).strip()
    if text == "" or text.lower() in {"none", "nan", "<na>"}:
        return None
    return Path(text).expanduser().resolve()


def _dedupe_paths(paths: Iterable[Path]) -> list[Path]:
    out, seen = [], set()
    for p in paths:
        try:
            rp = p.expanduser().resolve()
        except Exception:
            continue
        if str(rp) not in seen:
            out.append(rp)
            seen.add(str(rp))
    return out


def resolve_code_root() -> Path:
    override = _as_path_or_none(code_root_override)
    if override is not None:
        return override
    if NOTEBOOK_CWD.name == risk_averse_mutation_folder_name:
        return NOTEBOOK_CWD.parent
    if (NOTEBOOK_CWD / risk_averse_mutation_folder_name).exists():
        return NOTEBOOK_CWD
    if (NOTEBOOK_CWD.parent / risk_averse_mutation_folder_name).exists():
        return NOTEBOOK_CWD.parent
    return NOTEBOOK_CWD


def resolve_ra_dir(code_root: Path) -> Path:
    override = _as_path_or_none(risk_averse_mutation_dir_override)
    if override is not None:
        return override
    candidates = [NOTEBOOK_CWD, code_root / risk_averse_mutation_folder_name, NOTEBOOK_CWD / risk_averse_mutation_folder_name, NOTEBOOK_CWD.parent / risk_averse_mutation_folder_name]
    for c in _dedupe_paths(candidates):
        if (c / risk_output_base_folder_name).exists():
            return c
    return (code_root / risk_averse_mutation_folder_name).resolve()


CODE_ROOT = resolve_code_root()
RA_DIR = resolve_ra_dir(CODE_ROOT)
RISK_OUTPUT_BASE = _as_path_or_none(risk_output_base_dir_override) or (RA_DIR / risk_output_base_folder_name).resolve()
ANALYSIS_DIR = RISK_OUTPUT_BASE / analysis_output_folder_name
TABLE_DIR = ANALYSIS_DIR / "fe_distribution_tables"
FIGURE_DIR = ANALYSIS_DIR / "fe_distribution_figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Risk output base:", RISK_OUTPUT_BASE)
print("FE table dir    :", TABLE_DIR)
print("FE figure dir   :", FIGURE_DIR)

In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


In [ ]:
# ============================================================
# SECTION 4. INPUT LOADING HELPERS
# ============================================================

def _num(s):
    return pd.to_numeric(s, errors="coerce")


def risk_mode(lambda_s, lambda_b) -> str:
    ls = float(lambda_s) if pd.notna(lambda_s) else 0.0
    lb = float(lambda_b) if pd.notna(lambda_b) else 0.0
    eps = 1e-10
    if abs(ls) <= eps and abs(lb) <= eps:
        return "Risk neutral"
    if abs(ls) > eps and abs(lb) <= eps:
        return "Seller risk-averse"
    if abs(ls) <= eps and abs(lb) > eps:
        return "Buyer risk-averse"
    if abs(ls) > eps and abs(lb) > eps:
        return "Joint risk-averse"
    return "Other"


def normalize_family_value(value) -> str:
    text = str(value).strip()
    low = text.lower().replace("-", "_").replace(" ", "_")
    if low in family_display_labels:
        return family_display_labels[low]
    if "shape" in low:
        return family_display_labels["shape"]
    if "basis" in low:
        return family_display_labels["basis"]
    if "load" in low or "buyer" in low:
        return family_display_labels["load_price"]
    if "cannibal" in low or "seller" in low:
        return family_display_labels["cannibalization"]
    if "baseline" in low:
        return "Baseline"
    return text


def target_shift_label(value) -> str:
    if pd.isna(value):
        return "NA"
    v = float(value)
    if abs(v) < 1e-12:
        return "0"
    return f"{v:+.2f}"


def _parse_risk_from_folder(folder: Path) -> dict:
    name = folder.name
    out = {"risk_folder": name, "risk_setting_order": np.nan, "risk_label": name, "lambda_s": np.nan, "lambda_b": np.nan}
    m_order = re.match(r"^(\d+)_", name)
    if m_order:
        out["risk_setting_order"] = int(m_order.group(1))
    m = re.search(r"^(?:\d+_)?(?P<label>.*?)__lambdaS_(?P<ls>[^_]+)__lambdaB_(?P<lb>[^_]+)$", name)
    if m:
        out["risk_label"] = m.group("label")
        out["lambda_s"] = float(m.group("ls").replace("m", "-").replace("p", "."))
        out["lambda_b"] = float(m.group("lb").replace("m", "-").replace("p", "."))
    return out


def discover_hourly_fe_indices() -> pd.DataFrame:
    records = []
    if not RISK_OUTPUT_BASE.exists():
        raise FileNotFoundError(f"Risk output base does not exist: {RISK_OUTPUT_BASE}")
    for folder in sorted([p for p in RISK_OUTPUT_BASE.iterdir() if p.is_dir()]):
        idx_path = folder / "Hourly_FE_Index.csv"
        if not idx_path.exists():
            continue
        meta = _parse_risk_from_folder(folder)
        meta["output_root"] = str(folder)
        meta["hourly_fe_index_path"] = str(idx_path)
        records.append(meta)
    if not records:
        raise FileNotFoundError(f"No Hourly_FE_Index.csv files found under: {RISK_OUTPUT_BASE}")
    return pd.DataFrame(records)


def resolve_fe_file(path_value, output_root: Path) -> Optional[Path]:
    if path_value is None or (isinstance(path_value, float) and pd.isna(path_value)):
        return None
    text = str(path_value).strip()
    if not text or text.lower() in {"nan", "none", "<na>"}:
        return None
    p = Path(text).expanduser()
    candidates = []
    if p.is_absolute():
        candidates.append(p)
        candidates.append(output_root / "Hourly_FE" / p.name)
    else:
        candidates += [output_root / p, output_root / "Hourly_FE" / p.name, RA_DIR / p, CODE_ROOT / p, NOTEBOOK_CWD / p]
    for c in _dedupe_paths(candidates):
        if c.exists():
            return c
    return None


def load_hourly_fe_index() -> pd.DataFrame:
    inv = discover_hourly_fe_indices()
    frames = []
    for _, rec in inv.iterrows():
        idx = read_csv_optimized(rec["hourly_fe_index_path"])
        for col in ["risk_label", "lambda_s", "lambda_b", "risk_setting_order"]:
            if col not in idx.columns or idx[col].isna().all():
                idx[col] = rec.get(col, np.nan)
        idx["output_root"] = rec["output_root"]
        frames.append(idx)
    out = pd.concat(frames, ignore_index=True, sort=False)
    out["match_id"] = _num(out["match_id"]).astype("Int64")
    out["lambda_s"] = _num(out.get("lambda_s", 0)).fillna(0.0)
    out["lambda_b"] = _num(out.get("lambda_b", 0)).fillna(0.0)
    out["risk_mode"] = [risk_mode(s, b) for s, b in zip(out["lambda_s"], out["lambda_b"])]
    out["risk_intensity"] = np.maximum(out["lambda_s"].abs(), out["lambda_b"].abs())
    out["risk_label_display"] = out["risk_label"].astype(str) + " (λs=" + out["lambda_s"].map(lambda x: f"{x:.3g}") + ", λb=" + out["lambda_b"].map(lambda x: f"{x:.3g}") + ")"
    if "scenario_type" not in out.columns:
        out["scenario_type"] = np.where(out["scenario_name"].astype(str).str.lower().str.contains("baseline|no_mutation|no mutation"), "baseline", "mutation")
    if "mutation_family" not in out.columns:
        out["mutation_family"] = np.where(out["scenario_type"].eq("baseline"), "baseline", "unknown")
    out["mutation_family_normalized"] = out.get("mutation_family_label", out["mutation_family"]).map(normalize_family_value)
    out.loc[out["scenario_type"].astype(str).eq("baseline"), "mutation_family_normalized"] = "Baseline"
    if "target_physical_shift" in out.columns:
        out["target_shift_signed"] = _num(out["target_physical_shift"])
    elif "target_delta" in out.columns:
        out["target_shift_signed"] = _num(out["target_delta"])
    else:
        out["target_shift_signed"] = np.nan
    out.loc[out["scenario_type"].astype(str).eq("baseline"), "target_shift_signed"] = 0.0
    out["target_shift_label"] = out["target_shift_signed"].map(target_shift_label)
    return out


hourly_index_df = load_hourly_fe_index()
print("Loaded FE index rows:", len(hourly_index_df))
display(hourly_index_df.head())

In [ ]:
# ============================================================
# SECTION 5. SELECT FE FILES AND BUILD COMBINED FE TABLE
# ============================================================

def _numeric_filter(series, allowed, atol=1e-9):
    if allowed is None:
        return pd.Series(True, index=series.index)
    vals = [float(x) for x in (allowed if isinstance(allowed, (list, tuple, set)) else [allowed])]
    s = _num(series)
    keep = pd.Series(False, index=series.index)
    for v in vals:
        keep = keep | np.isclose(s, v, atol=atol)
    return keep


def filter_fe_index(idx: pd.DataFrame) -> pd.DataFrame:
    out = idx.copy()
    out = out.loc[out["match_id"].astype("Int64").eq(int(selected_match_id))].copy()
    if selected_risk_labels is not None:
        out = out.loc[out["risk_label"].astype(str).isin({str(x) for x in selected_risk_labels})].copy()
    if selected_risk_modes is not None:
        out = out.loc[out["risk_mode"].astype(str).isin({str(x) for x in selected_risk_modes})].copy()
    if selected_lambda_s_values is not None:
        out = out.loc[_numeric_filter(out["lambda_s"], selected_lambda_s_values)].copy()
    if selected_lambda_b_values is not None:
        out = out.loc[_numeric_filter(out["lambda_b"], selected_lambda_b_values)].copy()
    if selected_mutation_family is not None:
        fam = normalize_family_value(selected_mutation_family)
        is_base = out["scenario_type"].astype(str).eq("baseline")
        out = out.loc[is_base | out["mutation_family_normalized"].eq(fam)].copy()
    if selected_target_physical_shifts is not None:
        is_base = out["scenario_type"].astype(str).eq("baseline")
        out = out.loc[(is_base & bool(include_baseline)) | (~is_base & _numeric_filter(out["target_shift_signed"], selected_target_physical_shifts))].copy()
    elif not include_baseline:
        out = out.loc[~out["scenario_type"].astype(str).eq("baseline")].copy()
    return out.sort_values(["risk_setting_order", "scenario_type", "target_shift_signed"]).reset_index(drop=True)


def load_selected_fe_rows(fe_index: pd.DataFrame) -> pd.DataFrame:
    frames = []
    missing_files = []
    for _, rec in fe_index.iterrows():
        output_root = Path(str(rec["output_root"]))
        fe_path = resolve_fe_file(rec.get("hourly_fe_file", ""), output_root)
        if fe_path is None:
            missing_files.append(rec.to_dict())
            continue
        fe = read_csv_optimized(fe_path)
        for col in [
            "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity", "risk_setting_order",
            "match_id", "scenario_name", "scenario_type", "mutation_family_normalized", "target_shift_signed", "target_shift_label",
            "ppa_type", "profile_type", "decision_ppa_type", "decision_profile_type", "decision_strike_price_mwh", "decision_volume_mw",
        ]:
            if col in rec.index:
                fe[col] = rec[col]
        fe["fe_file"] = str(fe_path)
        frames.append(fe)
    if missing_files:
        missing = pd.DataFrame(missing_files)
        missing.to_csv(TABLE_DIR / "RA_Mutation_FE_Missing_Files.csv", index=False)
        print("Missing FE files:", len(missing), "details saved to RA_Mutation_FE_Missing_Files.csv")
    if not frames:
        raise FileNotFoundError("No FE CSV files could be loaded for the selected filters.")
    return pd.concat(frames, ignore_index=True, sort=False)


selected_fe_index_df = filter_fe_index(hourly_index_df)
if selected_fe_index_df.empty:
    raise ValueError("No Hourly_FE_Index rows remain after filters.")

fe_long_df = load_selected_fe_rows(selected_fe_index_df)

if save_fe_combined_table:
    fe_long_df.to_csv(TABLE_DIR / f"RA_Mutation_FE_Long_Match_{int(selected_match_id):04d}.csv", index=False)

print("Selected FE index rows:", len(selected_fe_index_df))
print("Loaded FE observations:", len(fe_long_df))
display(selected_fe_index_df[[c for c in ["risk_label", "lambda_s", "lambda_b", "match_id", "scenario_name", "target_shift_label", "hourly_fe_file"] if c in selected_fe_index_df.columns]].head(30))

In [ ]:
# ============================================================
# SECTION 6. FE SUMMARY TABLES
# ============================================================

def fe_summary_table(fe: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "risk_setting_order", "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity",
        "match_id", "scenario_name", "scenario_type", "mutation_family_normalized", "target_shift_signed", "target_shift_label",
        "decision_ppa_type", "decision_profile_type", "decision_strike_price_mwh", "decision_volume_mw",
    ]
    group_cols = [c for c in group_cols if c in fe.columns]
    rows = []
    for keys, sub in fe.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        row["n_replications"] = int(len(sub))
        for party, col in fe_columns.items():
            if col not in sub.columns:
                continue
            x = _num(sub[col]).replace([np.inf, -np.inf], np.nan).dropna()
            row[f"{party}_fe_mean"] = float(x.mean()) if not x.empty else np.nan
            row[f"{party}_fe_std"] = float(x.std(ddof=1)) if len(x) > 1 else 0.0 if len(x) == 1 else np.nan
            row[f"{party}_fe_q10"] = float(x.quantile(0.10)) if not x.empty else np.nan
            row[f"{party}_fe_q25"] = float(x.quantile(0.25)) if not x.empty else np.nan
            row[f"{party}_fe_median"] = float(x.median()) if not x.empty else np.nan
            row[f"{party}_fe_q75"] = float(x.quantile(0.75)) if not x.empty else np.nan
            row[f"{party}_fe_q90"] = float(x.quantile(0.90)) if not x.empty else np.nan
            lam = float(sub["lambda_s"].iloc[0]) if party == "seller" else float(sub["lambda_b"].iloc[0])
            row[f"{party}_fe_mean_plus_lambda_std"] = row[f"{party}_fe_mean"] + lam * row[f"{party}_fe_std"] if pd.notna(row[f"{party}_fe_mean"]) else np.nan
        if revenue_column in sub.columns:
            x = _num(sub[revenue_column]).replace([np.inf, -np.inf], np.nan).dropna()
            row[f"{revenue_column}_mean"] = float(x.mean()) if not x.empty else np.nan
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(["risk_setting_order", "target_shift_signed"]).reset_index(drop=True)
    return out


fe_summary_df = fe_summary_table(fe_long_df)
if save_fe_summary_table:
    fe_summary_df.to_csv(TABLE_DIR / f"RA_Mutation_FE_Summary_Match_{int(selected_match_id):04d}.csv", index=False, float_format="%.8g")

print("FE summary rows:", len(fe_summary_df))
display(fe_summary_df.head(30))

In [ ]:
# ============================================================
# SECTION 7. FE PLOTTING HELPERS
# ============================================================

def _safe_stem(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(text)).strip("_")


def _save_figure(fig, stem: str):
    saved = []
    if save_png:
        p = FIGURE_DIR / f"{stem}.png"
        fig.savefig(p, dpi=figure_dpi, bbox_inches="tight")
        saved.append(p)
    if save_pdf:
        p = FIGURE_DIR / f"{stem}.pdf"
        fig.savefig(p, bbox_inches="tight")
        saved.append(p)
    if save_svg:
        p = FIGURE_DIR / f"{stem}.svg"
        fig.savefig(p, bbox_inches="tight")
        saved.append(p)
    if saved:
        print("Saved", stem, "->", ", ".join(p.name for p in saved))


def _close_or_show(fig):
    if display_figures_in_notebook:
        display(fig)
    plt.close(fig)


def selected_parties() -> list[str]:
    if fe_party_to_plot == "both":
        return ["seller", "buyer"]
    return [fe_party_to_plot]


def scenario_order(df: pd.DataFrame) -> list[str]:
    meta = df[["scenario_name", "scenario_type", "target_shift_signed", "target_shift_label"]].drop_duplicates().copy()
    meta["_base"] = meta["scenario_type"].astype(str).eq("baseline").astype(int)
    meta["_shift"] = _num(meta["target_shift_signed"]).fillna(0)
    base = meta.loc[meta["_base"].eq(1)].sort_values("scenario_name")["scenario_name"].tolist()
    mut = meta.loc[meta["_base"].eq(0)].sort_values("_shift")["scenario_name"].tolist()
    return base + mut


def scenario_short_labels(df: pd.DataFrame) -> dict:
    labels = {}
    for scen, sub in df.groupby("scenario_name", dropna=False):
        row = sub.iloc[0]
        if str(row.get("scenario_type", "")) == "baseline":
            labels[scen] = "Baseline"
        else:
            labels[scen] = str(row.get("target_shift_label", scen))
    return labels

In [ ]:
# ============================================================
# SECTION 8. FE FIGURES
# ============================================================

def plot_fe_boxplots(fe: pd.DataFrame):
    if not make_boxplot_figure:
        return
    parties = selected_parties()
    risk_labels = fe[["risk_label_display", "risk_setting_order", "risk_intensity"]].drop_duplicates().sort_values(["risk_setting_order", "risk_intensity"])["risk_label_display"].tolist()
    for party in parties:
        col = fe_columns.get(party)
        if col not in fe.columns:
            print(f"Skipped {party} boxplot: {col} not found.")
            continue
        nrows = len(risk_labels)
        fig, axes = plt.subplots(nrows, 1, figsize=(default_fig_width, max(default_fig_height, 3.4 * nrows)), dpi=figure_dpi, squeeze=False)
        for r, risk in enumerate(risk_labels):
            ax = axes[r, 0]
            sub = fe.loc[fe["risk_label_display"].astype(str).eq(str(risk))].copy()
            scenarios = scenario_order(sub)
            labels = scenario_short_labels(sub)
            data = [_num(sub.loc[sub["scenario_name"].eq(scen), col]).dropna().to_numpy() for scen in scenarios]
            data = [d for d in data if len(d) > 0]
            plot_labels = [labels[scen] for scen in scenarios if len(_num(sub.loc[sub["scenario_name"].eq(scen), col]).dropna()) > 0]
            if data:
                ax.boxplot(data, labels=plot_labels, showfliers=boxplot_show_fliers)
            ax.set_title(str(risk), fontsize=panel_title_fontsize)
            ax.set_ylabel(f"{party} FE", fontsize=axis_label_fontsize)
            ax.tick_params(axis="x", labelrotation=x_tick_rotation, labelsize=tick_label_fontsize)
            ax.tick_params(axis="y", labelsize=tick_label_fontsize)
            ax.grid(True, axis="y", alpha=0.25)
        fig.suptitle(f"Contract-period {party} FE distributions, match {int(selected_match_id):04d}", fontsize=panel_title_fontsize)
        fig.tight_layout()
        _save_figure(fig, f"fig_fe_boxplot_match_{int(selected_match_id):04d}_{party}")
        _close_or_show(fig)


def ecdf_values(x):
    x = np.sort(np.asarray(pd.to_numeric(pd.Series(x), errors="coerce").dropna(), dtype=float))
    if len(x) == 0:
        return x, x
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y


def plot_fe_ecdf(fe: pd.DataFrame):
    if not make_ecdf_figure:
        return
    parties = selected_parties()
    risk_labels = fe[["risk_label_display", "risk_setting_order", "risk_intensity"]].drop_duplicates().sort_values(["risk_setting_order", "risk_intensity"])["risk_label_display"].tolist()
    for party in parties:
        col = fe_columns.get(party)
        if col not in fe.columns:
            continue
        nrows = len(risk_labels)
        fig, axes = plt.subplots(nrows, 1, figsize=(default_fig_width, max(default_fig_height, 3.4 * nrows)), dpi=figure_dpi, squeeze=False)
        for r, risk in enumerate(risk_labels):
            ax = axes[r, 0]
            sub = fe.loc[fe["risk_label_display"].astype(str).eq(str(risk))].copy()
            labels = scenario_short_labels(sub)
            for scen in scenario_order(sub):
                x, y = ecdf_values(sub.loc[sub["scenario_name"].eq(scen), col])
                if len(x) == 0:
                    continue
                ax.plot(x, y, linewidth=ecdf_linewidth, label=labels.get(scen, scen))
            ax.set_title(str(risk), fontsize=panel_title_fontsize)
            ax.set_xlabel(f"{party} contract-period FE", fontsize=axis_label_fontsize)
            ax.set_ylabel("ECDF", fontsize=axis_label_fontsize)
            ax.grid(True, alpha=0.25)
            ax.tick_params(labelsize=tick_label_fontsize)
            ax.legend(fontsize=legend_fontsize, frameon=True)
        fig.suptitle(f"Contract-period {party} FE ECDFs, match {int(selected_match_id):04d}", fontsize=panel_title_fontsize)
        fig.tight_layout()
        _save_figure(fig, f"fig_fe_ecdf_match_{int(selected_match_id):04d}_{party}")
        _close_or_show(fig)


def plot_mean_std_summary(summary: pd.DataFrame):
    if not make_mean_std_figure:
        return
    parties = selected_parties()
    for party in parties:
        mean_col = f"{party}_fe_mean"
        std_col = f"{party}_fe_std"
        util_col = f"{party}_fe_mean_plus_lambda_std"
        if mean_col not in summary.columns:
            continue
        risk_labels = summary[["risk_label_display", "risk_setting_order", "risk_intensity"]].drop_duplicates().sort_values(["risk_setting_order", "risk_intensity"])["risk_label_display"].tolist()
        fig, axes = plt.subplots(len(risk_labels), 1, figsize=(default_fig_width, max(default_fig_height, 3.3 * len(risk_labels))), dpi=figure_dpi, squeeze=False)
        for r, risk in enumerate(risk_labels):
            ax = axes[r, 0]
            sub = summary.loc[summary["risk_label_display"].astype(str).eq(str(risk))].sort_values("target_shift_signed")
            xlabels = ["Baseline" if str(t) == "0" else str(t) for t in sub["target_shift_label"]]
            x = np.arange(len(sub))
            ax.errorbar(x, sub[mean_col], yerr=sub[std_col], marker="o", capsize=3, label="mean ± sd")
            if util_col in sub.columns:
                ax.plot(x, sub[util_col], marker="s", linestyle="--", label="mean + λ·sd")
            ax.set_xticks(x)
            ax.set_xticklabels(xlabels, fontsize=tick_label_fontsize)
            ax.set_title(str(risk), fontsize=panel_title_fontsize)
            ax.set_ylabel(f"{party} FE", fontsize=axis_label_fontsize)
            ax.grid(True, axis="y", alpha=0.25)
            ax.legend(fontsize=legend_fontsize, frameon=True)
        axes[-1, 0].set_xlabel("Target physical shift", fontsize=axis_label_fontsize)
        fig.suptitle(f"{party.title()} FE mean, risk term, and dispersion, match {int(selected_match_id):04d}", fontsize=panel_title_fontsize)
        fig.tight_layout()
        _save_figure(fig, f"fig_fe_mean_std_match_{int(selected_match_id):04d}_{party}")
        _close_or_show(fig)


if make_figures:
    plot_fe_boxplots(fe_long_df)
    plot_fe_ecdf(fe_long_df)
    plot_mean_std_summary(fe_summary_df)

print("FE distribution analysis complete.")
print(f"Tables : {TABLE_DIR}")
print(f"Figures: {FIGURE_DIR}")